# BLAIR-Style Semantic Retrieval Baseline

This notebook implements the BLAIR-style semantic retrieval baseline for
query-conditioned product recommendation.

The baseline represents each recommendation request using only the generated
synthetic query. The query is encoded using the same E5-base-v2 semantic
encoder used to represent the Amazon Electronics product catalogue.

Retrieval is performed using the pre-built exact FAISS `IndexFlatIP` index
containing all products in the Amazon Electronics catalogue.

The baseline pipeline is:

Synthetic Query
→ E5 Query Embedding
→ Exact FAISS Search
→ Top-K Product Recommendations

The user preference profile is deliberately excluded from this baseline.

The resulting recommendations will later be compared with the proposed
profile-conditioned method using the same encoder, product representations,
FAISS index, evaluation examples, and retrieval settings.

### Live Demonstration — Query-Only Semantic Retrieval

The baseline retrieves products using only the current natural-language query. The query is encoded with E5-base-v2 and compared against the full product catalogue using FAISS.

In [2]:
import pandas as pd

# Load the saved query-only retrieval results
blair = pd.read_parquet(
    "/users/adhp511/query-conditioned-recommender/data/generated/blair_baseline_retrievals.parquet"
)

# Load the full product catalogue
catalogue_df = pd.read_parquet(
    "/users/adhp511/query-conditioned-recommender/data/generated/full_product_catalogue.parquet"
)

print(f"Baseline retrieval examples: {len(blair):,}")
print(f"Catalogue products: {len(catalogue_df):,}")

Baseline retrieval examples: 5,559
Catalogue products: 1,610,012


In [3]:
# Selected live-demo example
demo_sample_id = 1383

baseline_demo = blair[
    blair["sample_id"] == demo_sample_id
].iloc[0]

print("QUERY")
print(baseline_demo["synthetic_query"])

print("\nGROUND-TRUTH ASIN")
print(baseline_demo["ground_truth_asin"])

QUERY
lightning cable for iPhone 12

GROUND-TRUTH ASIN
B082X666WZ


In [4]:
# Display the Top-10 query-only results

retrieved_asins = baseline_demo["retrieved_asins"]
retrieval_scores = baseline_demo["retrieval_scores"]

print("QUERY-ONLY — TOP 10 RESULTS")
print("=" * 80)

for rank, (asin, score) in enumerate(
    zip(retrieved_asins[:10], retrieval_scores[:10]),
    start=1
):
    product = catalogue_df[
        catalogue_df["parent_asin"] == asin
    ].iloc[0]

    print(f"{rank:2d}. {product['title']}")
    print(f"    ASIN: {asin} | Score: {score:.4f}")

QUERY-ONLY — TOP 10 RESULTS
 1. iPhone 12 Charger Cable,IQIYI USB C to Lightning Cable Cord 3.3 ft [ Apple MFi Certified] for iPhone 12/Pro/Max/Mini/XS/XR/XS Max / 8/8 Plus, Supports 20w Power Delivery Fast Charging Syncing
    ASIN: B07RQ5SB4H | Score: 0.9023
 2. Retractable Lightning Cable [Apple MFi Certified], 3.3 Feet, 8 Pin to USB Sync Cable Charging Cord for iPhone 12, 12 Pro, 12 Pro Max, 12 Mini, 11, 11 Pro, 11 Pro Max, X Xr Xs Max, iPadi, iPod
    ASIN: B00IFVTQLK | Score: 0.9021
 3. Retractable Lightning Cable [Apple MFi Certified], 3.3 Feet, 8 Pin to USB Sync Cable Charging Cord for iPhone 12, 12 Pro, 12 Pro Max, 12 Mini, 11, 11 Pro, 11 Pro Max, X Xr Xs Max, iPadi, iPod
    ASIN: B00GLMAERU | Score: 0.9021
 4. j5create USB C to Lightning Cable for iPhone 12 (Black) - 4ft [Apple MFi Certified] Right Angle Plug Braided Cord for iPhone 12/12 Mini/12 Pro/11 Pro Max/X/XS/XR/8, iPad, AirPods Pro
    ASIN: B08X6W1WHX | Score: 0.9015
 5. USB C to Lightning Cable 2 Pack 6.6FT, Apple 

In [5]:
# Check whether the exact ground truth was retrieved

ground_truth = baseline_demo["ground_truth_asin"]

rank = (
    retrieved_asins.index(ground_truth) + 1
    if ground_truth in retrieved_asins
    else None
)

print("\n" + "=" * 80)

if rank is None:
    print("GROUND-TRUTH RESULT: NOT IN TOP-50")
else:
    print(f"GROUND-TRUTH RESULT: RANK {rank}")


GROUND-TRUTH RESULT: NOT IN TOP-50


## 1. Imports and Project Paths

Load the libraries required for semantic query encoding, FAISS retrieval,
data handling, and filesystem access.

All models and retrieval assets are loaded from the local Hyperion
environment; no external model download is required.

In [1]:
# Imports and project paths

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import torch
import faiss

from transformers import AutoTokenizer, AutoModel


# Project directories
PROJECT_ROOT = Path("/users/adhp511/query-conditioned-recommender")
DATA_DIR = PROJECT_ROOT / "data"
GENERATED_DIR = DATA_DIR / "generated"

# Local E5 model
E5_MODEL_PATH = Path.home() / "models" / "e5-base-v2"

# Shared retrieval assets created in Notebook 08
PRODUCT_CATALOGUE_PATH = GENERATED_DIR / "full_product_catalogue.parquet"
FAISS_INDEX_PATH = GENERATED_DIR / "e5_faiss_product_index.bin"

# Evaluation data
EVALUATION_PATH = GENERATED_DIR / "proposed_model_data.parquet"

# Retrieval settings
TOP_K_VALUES = [1, 5, 10, 20, 50]
MAX_LENGTH = 256
BATCH_SIZE = 256

# Computation device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("E5 model:", E5_MODEL_PATH)
print("Product catalogue:", PRODUCT_CATALOGUE_PATH)
print("FAISS index:", FAISS_INDEX_PATH)
print("Evaluation data:", EVALUATION_PATH)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/users/adhp511/.pyenv/versions/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /users/adhp511/query-conditioned-recommender
E5 model: /users/adhp511/models/e5-base-v2
Product catalogue: /users/adhp511/query-conditioned-recommender/data/generated/full_product_catalogue.parquet
FAISS index: /users/adhp511/query-conditioned-recommender/data/generated/e5_faiss_product_index.bin
Evaluation data: /users/adhp511/query-conditioned-recommender/data/generated/proposed_model_data.parquet
Device: cuda
GPU: Quadro RTX 8000


## 2. Load the E5 Semantic Encoder

Load the locally stored `intfloat/e5-base-v2` tokenizer and encoder.

The model is used only for inference. It will encode the synthetic queries
using E5's query representation format:

`query: [synthetic query]`

The product embeddings in the shared FAISS index were created using the
corresponding E5 passage representation:

`passage: [product text]`

Using the same encoder for both sides keeps the query and product embeddings
in the same semantic space.

In [2]:
# Load E5-base-v2 locally

print("Loading E5 from:", E5_MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(
    E5_MODEL_PATH,
    local_files_only=True
)

e5_model = AutoModel.from_pretrained(
    E5_MODEL_PATH,
    local_files_only=True
)

e5_model = e5_model.to(DEVICE)
e5_model.eval()

EMBEDDING_DIM = e5_model.config.hidden_size

print("\nModel loaded successfully.")
print("Embedding dimension:", EMBEDDING_DIM)
print("Device:", DEVICE)
print("Model in evaluation mode:", not e5_model.training)

Loading E5 from: /users/adhp511/models/e5-base-v2


Loading weights: 100%|██████████████████████████████████████████████| 199/199 [00:00<00:00, 8384.23it/s]



Model loaded successfully.
Embedding dimension: 768
Device: cuda
Model in evaluation mode: True


## 3. Load the Shared FAISS Product Index

Load the exact FAISS index created in Notebook 08.

The index contains the E5 passage embeddings for all products in the full
Amazon Electronics catalogue. No product filtering or candidate generation
is applied for this baseline.

In [3]:
# Load the shared full-catalogue FAISS index

print("Loading FAISS index...")
print(FAISS_INDEX_PATH)

faiss_index = faiss.read_index(
    str(FAISS_INDEX_PATH)
)

print("\nFAISS index loaded successfully.")
print("Index type:", type(faiss_index).__name__)
print("Number of products:", f"{faiss_index.ntotal:,}")
print("Embedding dimension:", faiss_index.d)

assert faiss_index.ntotal == 1_610_012
assert faiss_index.d == EMBEDDING_DIM

print("\n✓ FAISS index is ready for baseline retrieval.")

Loading FAISS index...
/users/adhp511/query-conditioned-recommender/data/generated/e5_faiss_product_index.bin

FAISS index loaded successfully.
Index type: IndexFlatIP
Number of products: 1,610,012
Embedding dimension: 768

✓ FAISS index is ready for baseline retrieval.


## 4. Load the Baseline Evaluation Dataset

Load the 5,559 evaluation examples used for the retrieval experiment.

For the BLAIR-style baseline, only the synthetic query and ground-truth
product identifier are used.

The user preference profile is intentionally excluded so that the baseline
represents query-only semantic retrieval.

In [4]:
# Load evaluation data for the BLAIR-style baseline

evaluation_data = pd.read_parquet(
    EVALUATION_PATH,
    columns=[
        "sample_id",
        "user_id",
        "synthetic_query",
        "ground_truth_asin"
    ]
)

print("Evaluation shape:", evaluation_data.shape)
print("Columns:", evaluation_data.columns.tolist())

print("\nMissing values:")
print(evaluation_data.isna().sum())

print("\nFirst evaluation example:")
display(evaluation_data.head(1))

assert evaluation_data.shape[0] == 5559
assert evaluation_data["sample_id"].nunique() == 5559
assert evaluation_data["synthetic_query"].notna().all()
assert evaluation_data["ground_truth_asin"].notna().all()

print("\n✓ Baseline evaluation dataset loaded and validated.")

Evaluation shape: (5559, 4)
Columns: ['sample_id', 'user_id', 'synthetic_query', 'ground_truth_asin']

Missing values:
sample_id            0
user_id              0
synthetic_query      0
ground_truth_asin    0
dtype: int64

First evaluation example:


,sample_id,user_id,synthetic_query,ground_truth_asin
0,0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,Looking for a high-speed micro sd card for my ...,B007VKII6A



✓ Baseline evaluation dataset loaded and validated.


## 5. Define the E5 Query-Embedding Function

Define a reusable function that converts synthetic queries into normalised
768-dimensional E5 query embeddings.

For the BLAIR-style baseline, the input is only the synthetic query:

`query: [synthetic query]`

The function uses mean pooling with the attention mask followed by L2
normalisation, matching the product-embedding procedure used in Notebook 08.

In [5]:
# E5 query-embedding function for the baseline

def encode_queries(
    queries,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
):
    """
    Encode a list of queries using E5-base-v2.

    E5 query format:
        query: [query text]

    Returns:
        np.ndarray of shape (n_queries, 768), float32,
        with L2-normalised embeddings.
    """

    all_embeddings = []

    for start in range(0, len(queries), batch_size):
        batch_queries = queries[start:start + batch_size]

        # E5 query prefix
        inputs = tokenizer(
            [f"query: {query}" for query in batch_queries],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = e5_model(**inputs)

        # Mean pooling using the attention mask
        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_embeddings = outputs.last_hidden_state * attention_mask
        sum_embeddings = masked_embeddings.sum(dim=1)
        sum_mask = attention_mask.sum(dim=1).clamp(min=1)

        embeddings = sum_embeddings / sum_mask

        # L2 normalisation
        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

        all_embeddings.append(
            embeddings.cpu().numpy().astype(np.float32)
        )

    return np.vstack(all_embeddings)

## 6. Test Baseline Query Encoding

Encode a small sample of real synthetic queries using the baseline query
encoder.

This test verifies that the function returns the expected 768-dimensional,
L2-normalised embeddings before running the complete evaluation set.

In [6]:
# Test the baseline query encoder

TEST_QUERY_COUNT = 8

test_queries = (
    evaluation_data["synthetic_query"]
    .head(TEST_QUERY_COUNT)
    .tolist()
)

test_query_embeddings = encode_queries(test_queries)

print("Queries encoded:", len(test_queries))
print("Embedding shape:", test_query_embeddings.shape)
print("Embedding dtype:", test_query_embeddings.dtype)

test_norms = np.linalg.norm(
    test_query_embeddings,
    axis=1
)

print("\nL2 norms:")
print(test_norms)

print("\nFirst query:")
print(test_queries[0])

assert test_query_embeddings.shape == (
    TEST_QUERY_COUNT,
    EMBEDDING_DIM
)

assert test_query_embeddings.dtype == np.float32

assert np.allclose(
    test_norms,
    1.0,
    atol=1e-5
)

assert np.isfinite(test_query_embeddings).all()

print("\n✓ Baseline query-encoding test passed.")

Queries encoded: 8
Embedding shape: (8, 768)
Embedding dtype: float32

L2 norms:
[1.         1.         1.         1.         0.99999994 1.
 1.         1.        ]

First query:
Looking for a high-speed micro sd card for my action camera

✓ Baseline query-encoding test passed.


## 7. Encode All Baseline Queries

Encode all 5,559 synthetic evaluation queries using the BLAIR-style
query-only representation.

Each query is encoded independently as:

`query: [synthetic query]`

The resulting matrix contains one 768-dimensional normalised embedding per
evaluation example.

These embeddings will then be searched against the shared full-catalogue
FAISS index.

In [7]:
# Encode all synthetic queries for the BLAIR-style baseline

all_baseline_queries = (
    evaluation_data["synthetic_query"]
    .astype(str)
    .tolist()
)

print("Encoding baseline queries...")
print("Number of queries:", len(all_baseline_queries))
print("Batch size:", BATCH_SIZE)

encoding_start = time.time()

baseline_query_embeddings = encode_queries(
    all_baseline_queries,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

encoding_time = time.time() - encoding_start

print("\nBaseline query encoding complete.")
print("Embedding shape:", baseline_query_embeddings.shape)
print("Embedding dtype:", baseline_query_embeddings.dtype)
print("Encoding time:", round(encoding_time, 2), "seconds")

norms = np.linalg.norm(
    baseline_query_embeddings,
    axis=1
)

print("Average L2 norm:", norms.mean())
print("Minimum L2 norm:", norms.min())
print("Maximum L2 norm:", norms.max())

assert baseline_query_embeddings.shape == (
    len(evaluation_data),
    EMBEDDING_DIM
)

assert baseline_query_embeddings.dtype == np.float32
assert np.isfinite(baseline_query_embeddings).all()
assert np.allclose(norms, 1.0, atol=1e-5)

print("\n✓ All baseline query embeddings validated.")

Encoding baseline queries...
Number of queries: 5559
Batch size: 256

Baseline query encoding complete.
Embedding shape: (5559, 768)
Embedding dtype: float32
Encoding time: 2.23 seconds
Average L2 norm: 1.0
Minimum L2 norm: 0.9999999
Maximum L2 norm: 1.0000001

✓ All baseline query embeddings validated.


## 8. Retrieve Top-50 Products for the BLAIR-Style Baseline

Search all 5,559 baseline query embeddings against the complete
1,610,012-product FAISS index.

For each evaluation example, retrieve the 50 highest-scoring products using
exact inner-product similarity.

Because both query and product embeddings are L2-normalised, the inner
product corresponds to cosine similarity.

No product-type filtering, constraint extraction, candidate restriction, or
reranking is applied.

In [8]:
# Retrieve Top-50 products for every baseline query

RETRIEVAL_K = 50

print("Starting baseline retrieval...")
print("Queries:", f"{len(baseline_query_embeddings):,}")
print("Catalogue products:", f"{faiss_index.ntotal:,}")
print("Top-K:", RETRIEVAL_K)

retrieval_start = time.time()

baseline_scores, baseline_indices = faiss_index.search(
    baseline_query_embeddings,
    RETRIEVAL_K
)

retrieval_time = time.time() - retrieval_start

print("\nBaseline retrieval complete.")
print("Scores shape:", baseline_scores.shape)
print("Indices shape:", baseline_indices.shape)
print("Retrieval time:", round(retrieval_time, 2), "seconds")

assert baseline_scores.shape == (
    len(evaluation_data),
    RETRIEVAL_K
)

assert baseline_indices.shape == (
    len(evaluation_data),
    RETRIEVAL_K
)

assert np.isfinite(baseline_scores).all()
assert np.all(baseline_indices >= 0)
assert np.all(baseline_indices < faiss_index.ntotal)

print("\n✓ Baseline retrieval completed successfully.")

Starting baseline retrieval...
Queries: 5,559
Catalogue products: 1,610,012
Top-K: 50

Baseline retrieval complete.
Scores shape: (5559, 50)
Indices shape: (5559, 50)
Retrieval time: 24.26 seconds

✓ Baseline retrieval completed successfully.


## 9. Map FAISS Results to Product Identifiers

FAISS returns integer row indices rather than product identifiers.

Because the FAISS index was constructed using the product embeddings in the
same row order as the product catalogue, each returned index maps directly to
the corresponding catalogue row.

Convert the Top-50 FAISS indices into product identifiers and inspect the
retrieved products for several evaluation examples.

The ground-truth product is not used to alter the ranking; it is only shown
for later evaluation.

In [9]:
# Map FAISS indices back to catalogue product identifiers

catalogue_mapping = pd.read_parquet(
    PRODUCT_CATALOGUE_PATH,
    columns=["parent_asin", "title", "store"]
)

print("Catalogue mapping shape:", catalogue_mapping.shape)

assert len(catalogue_mapping) == faiss_index.ntotal

# Convert FAISS indices into product identifiers
baseline_retrieved_asins = catalogue_mapping.iloc[
    baseline_indices.ravel()
]["parent_asin"].to_numpy().reshape(
    baseline_indices.shape
)

print("Retrieved ASIN matrix shape:", baseline_retrieved_asins.shape)

assert baseline_retrieved_asins.shape == baseline_indices.shape

# Inspect the first 3 evaluation examples
for example_idx in range(3):
    print("\n" + "=" * 80)
    print("Evaluation example:", example_idx)
    print("Query:", evaluation_data.iloc[example_idx]["synthetic_query"])
    print(
        "Ground truth:",
        evaluation_data.iloc[example_idx]["ground_truth_asin"]
    )

    inspection = pd.DataFrame({
        "rank": np.arange(1, 11),
        "similarity": baseline_scores[example_idx, :10],
        "parent_asin": baseline_retrieved_asins[example_idx, :10],
        "title": catalogue_mapping.iloc[
            baseline_indices[example_idx, :10]
        ]["title"].to_numpy(),
        "store": catalogue_mapping.iloc[
            baseline_indices[example_idx, :10]
        ]["store"].to_numpy(),
    })

    display(inspection)

print("\n✓ FAISS indices successfully mapped to catalogue products.")

Catalogue mapping shape: (1610012, 3)
Retrieved ASIN matrix shape: (5559, 50)

Evaluation example: 0
Query: Looking for a high-speed micro sd card for my action camera
Ground truth: B007VKII6A


,rank,similarity,parent_asin,title,store
0,1,0.906942,B08QW2KRC7,32GB High Speed Micro SD Card for Action Camer...,ABTOCAR
1,2,0.883489,B078PBL7L7,Micro SD Card 256GB High Speed Micro SDXC Clas...,
2,3,0.882886,B09MW9947B,512GB Micro SD High Speed Transfer Micro SD Ca...,HUNYEIZ
3,4,0.882382,B0CB3XPVXX,Micro SD Card 512GB Micro SD Memory Card 512GB...,SUPERDUODUO
4,5,0.881829,B09NBYN9VR,512GB microSDXC High Speed Transfer Micro SD C...,HUNYEIZ
5,6,0.880240,B09B997QK7,"128GB Micro SD Card, KODAK MicroSDXC Memory Ca...",KODAK
6,7,0.877296,B0786HF5VZ,32GB High Speed Micro Memory Card Transfer Spe...,Jusinhel
7,8,0.876014,B07NZ4HR7Q,Newwings 32GB High Speed Micro SD Card for Act...,Newwings
8,9,0.875576,B0BGBFQ4FW,512GB Micro SD High Speed Memory Card Class 10...,MORIC
9,10,0.875015,B0C8SWN464,Micro SD Card 512GB Mini SD Card for GOPRO Cam...,nuiflash



Evaluation example: 1
Query: looking for a good quality internal hard drive to backup my important files
Ground truth: B07454F4JH


,rank,similarity,parent_asin,title,store
0,1,0.844733,B000XSFUE0,Acomdata HD500FPMM-72 500GB MiniPAL Backup Har...,Acomdata
1,2,0.842734,B01M2WM16G,Marshal 1TB Internal Hard Disc Drive MAL21000S...,Amazon Renewed
2,3,0.841952,B09G8JLRZ5,Seagate IronWolf Pro ST16000NE000 16 TB Hard D...,Amazon Renewed
3,4,0.841015,B004U092JG,PQI H567L 500GB External Hard Disk Drive (656L...,PQI
4,5,0.840777,B092V8ZVZ4,SUYING HDD External Hard Drive 2tb / 1tb / 500...,SUYING
5,6,0.840155,B019EDXM30,White Label 6TB 7200RPM 64MB Cache SATA 6.0Gb/...,GoHardDrive
6,7,0.840041,B01B5ZWXU8,MasterStor Premium 160GB Slim Portable Hard Dr...,ORICO
7,8,0.839967,B01LXDF80N,2.5 Inch 500GB SATA 7mm Thickness Laptop Inter...,Mexone
8,9,0.839880,B0006FS24K,Seagate 9Y7274-557 300 GB Internal Ultra ATA H...,Seagate
9,10,0.839651,B009MJE6I2,PQI H551 500GB External Hard Disk Drive (6551-...,PQI



Evaluation example: 2
Query: looking for a good quality internal hard drive for my desktop
Ground truth: B00SMVLPIK


,rank,similarity,parent_asin,title,store
0,1,0.865863,B01KALOV08,Seagate 1TB Desktop HDD Hard Drive - Internal ...,Seagate
1,2,0.865863,B0142JJQOK,Seagate 1TB Desktop HDD Hard Drive - Internal ...,Seagate
2,3,0.858538,B014QQE0Y0,Toshiba 2TB Desktop 7200rpm Internal Hard Drive,Toshiba
3,4,0.856731,B01ICLGJ7Q,"Seagate BarraCuda 3TB 3.5"" Internal Desktop Ha...",Seagate
4,5,0.856588,B00VPKPQK8,Seagate Cheetah 15K Hard Drive - Internal (ST3...,Seagate
5,6,0.856224,B01LXDF80N,2.5 Inch 500GB SATA 7mm Thickness Laptop Inter...,Mexone
6,7,0.855973,B07GJX22H6,"Seagate 1TB 3.5"" SATA HDD 7200RPM Internal Des...",Oemgenuine
7,8,0.855636,B09CDN1XG1,KingSpec 2.5 Inch SATA SSD 128GB SATA3 Interna...,KingSpec
8,9,0.855303,B0897C69SX,KingShark SSD M.2 2260 500GB Internal Solid St...,KingShark
9,10,0.854968,B09GT8RK6K,MaxDigitalData (MD16000GSA25672) 16TB 7200RPM ...,Amazon Renewed



✓ FAISS indices successfully mapped to catalogue products.


## 10. Save BLAIR-Style Baseline Retrievals

Save the Top-50 retrieval results for all 5,559 evaluation examples.

For each example, store:

- `sample_id`
- `user_id`
- `synthetic_query`
- `ground_truth_asin`
- the ordered list of 50 retrieved product identifiers
- the corresponding 50 similarity scores

The retrieval order is preserved, with position 1 representing the highest
similarity result.

These saved predictions will be used later by the evaluation notebook.

In [10]:
# Save BLAIR-style baseline retrieval results

BASELINE_RESULTS_PATH = (
    GENERATED_DIR / "blair_baseline_retrievals.parquet"
)

baseline_results = evaluation_data[
    [
        "sample_id",
        "user_id",
        "synthetic_query",
        "ground_truth_asin"
    ]
].copy()

baseline_results["retrieved_asins"] = [
    row.tolist()
    for row in baseline_retrieved_asins
]

baseline_results["retrieval_scores"] = [
    row.astype(float).tolist()
    for row in baseline_scores
]

baseline_results.to_parquet(
    BASELINE_RESULTS_PATH,
    index=False
)

print("Baseline retrievals saved successfully.")
print("Path:", BASELINE_RESULTS_PATH)
print("Rows:", len(baseline_results))
print("Columns:", baseline_results.columns.tolist())
print(
    "File size:",
    round(BASELINE_RESULTS_PATH.stat().st_size / (1024**2), 2),
    "MB"
)

# Validate saved structure
saved_check = pd.read_parquet(BASELINE_RESULTS_PATH)

print("\nSaved shape:", saved_check.shape)
print("Retrieved ASINs per example:",
      saved_check["retrieved_asins"].str.len().unique())
print("Scores per example:",
      saved_check["retrieval_scores"].str.len().unique())

assert len(saved_check) == 5559
assert saved_check["retrieved_asins"].str.len().eq(50).all()
assert saved_check["retrieval_scores"].str.len().eq(50).all()

print("\n✓ Baseline retrieval results saved and validated.")

Baseline retrievals saved successfully.
Path: /users/adhp511/query-conditioned-recommender/data/generated/blair_baseline_retrievals.parquet
Rows: 5559
Columns: ['sample_id', 'user_id', 'synthetic_query', 'ground_truth_asin', 'retrieved_asins', 'retrieval_scores']
File size: 3.79 MB

Saved shape: (5559, 6)
Retrieved ASINs per example: [50]
Scores per example: [50]

✓ Baseline retrieval results saved and validated.
